# OrderBot - Week 3 API Notebook
**CoreSmart GenAI Developer Course · Week 3**

OrderBot takes a natural-language order question, has GPT call a lookup tool,
and returns a structured answer with full telemetry. This notebook exercises
every endpoint and exercises every failure mode introduced in Week 3.

Each endpoint is shown two ways:
- **cURL (Windows cmd)** - `%%cmd` cell magic, Windows double-quote syntax
- **Python** - `requests` library, works everywhere

---
### Before you start
1. Server running: `uvicorn app.main:app --reload`
2. `.env` file in `w03v03/` with `OPENAI_API_KEY=sk-...`
3. Run the **Setup** cell below once.

> **Windows note:** All curl cells use `%%cmd` with `\"` to escape inner quotes.

In [1]:
# Setup -- run this cell first
import requests, json, textwrap

BASE = 'http://localhost:8000'

# ASCII only -- no em dashes or Unicode (breaks Windows cmd curl)
DEMO_NOTES = 'Where is my order ORD-1042? It was supposed to arrive Monday.'

# Extra demos for multi-model comparison
DEMO = {
    'order':   'Where is my order ORD-1042? It was supposed to arrive Monday.',
    'refund':  'I want a refund on order ORD-2222 -- it arrived damaged.',
    'unknown': 'order 42',
}

print('Setup complete.')
print('BASE:', BASE)

Setup complete.
BASE: http://localhost:8000


---
## 1 · Health Check - `GET /health`
Confirms the server is alive and shows which model is loaded.

> This section is identical across all weeks. Do not modify it.

In [2]:
%%cmd
curl -s http://localhost:8000/health

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 3/>curl -s http://localhost:8000/health
{"status":"ok","model":"gpt-5.4-mini-2026-03-17","models":{"openai":"gpt-5.4-mini-2026-03-17","nano":"gpt-5.4-nano"}}
week 3/>

In [3]:
# Health check -- Python
r = requests.get(f'{BASE}/health')
print('Status :', r.status_code)
print(json.dumps(r.json(), indent=2))

Status : 200
{
  "status": "ok",
  "model": "gpt-5.4-mini-2026-03-17",
  "models": {
    "openai": "gpt-5.4-mini-2026-03-17",
    "nano": "gpt-5.4-nano"
  }
}


---
## 2 · Order Lookup - `POST /lookup`

Sends a natural-language question to the model. The model calls `lookup_order_status`,
gets back the typed result, then produces a final plain-text answer.

Key concept: **four-message round-trip** - user message → tool_call → tool_result → final text.
The `tool_call_count` and `retry_count` fields expose the retry-with-correction layer.

Request body:
```json
{ "message": "string", "provider": "openai" | "nano" }
```

Response shape:
```json
{
  "answer": "string",
  "tool_call_count": 1,
  "retry_count": 0,
  "provider": "openai",
  "model": "gpt-5.4-mini-2026-03-17",
  "latency_ms": 312.4
}
```

> Requires `OPENAI_API_KEY` in `.env`.

In [4]:
%%cmd
curl -s -X POST http://localhost:8000/lookup -H "Content-Type: application/json" -d "{\"message\": \"Where is my order ORD-1042?\", \"provider\": \"openai\"}"

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 3/>curl -s -X POST http://localhost:8000/lookup -H "Content-Type: application/json" -d "{\"message\": \"Where is my order ORD-1042?\", \"provider\": \"openai\"}"
{"answer":"Your order ORD-1042 has shipped and is expected to arrive Friday by 8pm local time.","tool_call_count":1,"retry_count":0,"latency_ms":2750.0,"provider":"openai","model":"gpt-5.4-mini-2026-03-17"}
week 3/>

In [5]:
# POST /lookup -- Python
r = requests.post(f'{BASE}/lookup', json={'message': DEMO_NOTES, 'provider': 'openai'})
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    data = r.json()
    print('-- Answer --')
    print(textwrap.fill(data['answer'], 80))
    print()
    print('-- Telemetry --')
    print(f'  tool_calls : {data["tool_call_count"]}')
    print(f'  retries    : {data["retry_count"]}')
    print(f'  latency_ms : {data["latency_ms"]}ms')
    print(f'  model      : {data["model"]}')

-- Answer --
Your order ORD-1042 is shipped, and the current ETA is Friday by 8pm local.
Last update: 2026-06-05 23:33:02.708529

-- Telemetry --
  tool_calls : 1
  retries    : 0
  latency_ms : 2079.0ms
  model      : gpt-5.4-mini-2026-03-17


---
## 3 · Multi-Model Comparison - openai vs nano

Both providers call the same tool. Compare answer quality and latency.

Key concept: `provider` maps to a **pinned model version** in `config.py`.
Swapping providers never requires touching the tool or retry logic.

In [6]:
# Multi-model loop -- same input, both providers side by side
print(f'Input: "{DEMO_NOTES}"\n')
print(f'{"provider":<10} {"tool_calls":<12} {"retries":<10} {"latency":<12} model')
print('-' * 72)
for provider in ['openai', 'nano']:
    r = requests.post(f'{BASE}/lookup', json={'message': DEMO_NOTES, 'provider': provider})
    if r.status_code != 200:
        print(f'{provider:<10} ERROR: {r.json().get("detail", "?")}')
    else:
        d = r.json()
        print(f'{d["provider"]:<10} {d["tool_call_count"]:<12} {d["retry_count"]:<10} {d["latency_ms"]:<12.1f}ms  {d["model"]}')

Input: "Where is my order ORD-1042? It was supposed to arrive Monday."

provider   tool_calls   retries    latency      model
------------------------------------------------------------------------
openai     1            0          3734.0      ms  gpt-5.4-mini-2026-03-17
nano       1            0          3610.0      ms  gpt-5.4-nano


---
## 4 · Telemetry Headers

The server echoes `tool_call_count`, `retry_count`, and `latency_ms` as response headers.
These are useful for dashboards without parsing the body.

In [7]:
# Inspect response headers
r = requests.post(f'{BASE}/lookup', json={'message': DEMO_NOTES, 'provider': 'openai'})
for h in ['X-Tool-Call-Count', 'X-Retry-Count', 'X-Latency-Ms']:
    print(f'{h}: {r.headers.get(h, "(missing)")}')

X-Tool-Call-Count: 1
X-Retry-Count: 0
X-Latency-Ms: 2640.0


---
## 5 · Failure Mode - Empty Message (422)

Pydantic validates `message` with `min_length=1` before any API call is made.
An empty string returns **422** - no tokens spent.

> This failure pattern is identical across all weeks. Update the endpoint and payload only.

In [8]:
%%cmd
curl -s -X POST http://localhost:8000/lookup -H "Content-Type: application/json" -d "{\"message\": \"\"}"

Microsoft Windows [Version 10.0.26200.8524]
(c) Microsoft Corporation. All rights reserved.

week 3/>curl -s -X POST http://localhost:8000/lookup -H "Content-Type: application/json" -d "{\"message\": \"\"}"
{"detail":[{"type":"string_too_short","loc":["body","message"],"msg":"String should have at least 1 character","input":"","ctx":{"min_length":1}}]}
week 3/>

In [9]:
# Failure: empty message -- Python
r = requests.post(f'{BASE}/lookup', json={'message': ''})
print(f'Status: {r.status_code}  (expected 422 -- Pydantic rejects empty message, no API call made)')
print(json.dumps(r.json(), indent=2))

Status: 422  (expected 422 -- Pydantic rejects empty message, no API call made)
{
  "detail": [
    {
      "type": "string_too_short",
      "loc": [
        "body",
        "message"
      ],
      "msg": "String should have at least 1 character",
      "input": "",
      "ctx": {
        "min_length": 1
      }
    }
  ]
}


---
## 6 · Failure Mode - Retry-with-Correction (model argument mismatch)

When the model produces malformed tool arguments (e.g. `order_id=42` instead of
`order_id='ORD-42'`), the retry-with-correction loop fires. `retry_count > 0`
in the response indicates this happened.

The ambiguous message below intentionally leaves out the `ORD-` prefix to
push the model toward an integer interpretation on the first attempt.

In [10]:
# Trigger retry-with-correction by sending an ambiguous order reference
r = requests.post(f'{BASE}/lookup', json={'message': 'What is the status of order 42?', 'provider': 'openai'})
if r.status_code != 200:
    print(f'Error {r.status_code}:', r.json())
else:
    d = r.json()
    print('-- Answer --')
    print(textwrap.fill(d['answer'], 80))
    print()
    print(f'retry_count: {d["retry_count"]}  (> 0 means correction loop fired)')

-- Answer --
I need the full order ID in the required format (for example, `ORD-1042`).
Please send your order ID so I can look up the status. I need the full order ID
in the required format (for example, `ORD-1042`). Please send your order ID so I
can look up the status.

retry_count: 0  (> 0 means correction loop fired)


---
## 7 · Failure Mode - Transient Tool Failure (`tool_impl_transient_failed`)

The failures above are visible over HTTP. The next three are **below** the HTTP
surface - they live in the dispatcher, the retry decorator, and the tool loop.
We exercise them in-process, against the real `app/` modules, with the model
stubbed. No server, no API key, no tokens spent.

First, a tiny log handler so the structured `event` tag on each log record is
visible - that tag is the whole point of these failure modes.

**Transient tool failure.** `with_tool_retry` is scaffolded in `retry.py` and
**deliberately not applied** in the shipped code, because `lookup_order_status`
is a local function: it has no network, so it has no transient failure mode.
Here we simulate the real thing - a flaky orders service that times out once -
and watch tenacity absorb it.


In [ ]:
# In-process failure demos -- no server, no API key, no tokens.
import os, logging, httpx, json
os.environ.setdefault('OPENAI_API_KEY', 'sk-notebook-stub')   # stubs only; never sent

from app.retry import with_tool_retry
from app.tools import lookup_order_status, execute_tool

class TagHandler(logging.Handler):
    """Print the structured `event` tag that rides on each log record."""
    def emit(self, record):
        tag = getattr(record, 'event', None)
        if tag:
            print(f'  LOG  event={tag:<28} {record.getMessage()}')

root = logging.getLogger()
root.handlers = [TagHandler()]
root.setLevel(logging.INFO)

# A flaky version of the tool: times out on the first call, succeeds on the second.
attempts = {'n': 0}

@with_tool_retry
def flaky_lookup(order_id: str):
    attempts['n'] += 1
    if attempts['n'] == 1:
        raise httpx.TimeoutException('orders-service timed out')
    return lookup_order_status(order_id)

result = flaky_lookup('ORD-1042')
print()
print('attempts made :', attempts['n'], ' (tenacity backed off, then retried)')
print('final result  :', result.state)
print()
print('Note: the model never saw this failure. It happened BELOW the model, at the')
print('dispatcher layer, so retry_count in the HTTP response would still read 0.')
print('The only trace is the extra latency of that backoff -- and that log tag.')


---
## 8 · Failure Mode - Wrong Tool Choice (`wrong_tool_choice`)

The model asks for a tool that is not in the `TOOLS` registry. `execute_tool`
does **not** raise: it logs `event: wrong_tool_choice` (with the tools it does
know about) and returns a structured `unknown_tool` envelope the model can read.

With one tool in the registry this is nearly impossible. With six tools whose
descriptions have drifted together over a year of edits, it is a Tuesday.


In [ ]:
# The model calls a tool that does not exist
envelope = execute_tool('lookup_order_stats', {'order_id': 'ORD-1042'})   # note the typo
print()
print('envelope:', envelope)
print()
print('A rising wrong_tool_choice rate almost always means two tool DESCRIPTIONS')
print('have converged. Fix the descriptions, not the dispatcher.')


---
## 9 · Failure Mode - Runaway Loop (`loop_exceeded` -> HTTP 429)

`tool_loop_max_iterations` (5, in `config.py`) is the ceiling on tool calls per
user turn. If the model keeps asking for tools past that cap, `call_with_tools`
logs `event: loop_exceeded`, raises `LoopExceededError`, and `main.py` maps it
to **HTTP 429**. This is the runaway-agent guard: the difference between a bad
afternoon and a bad invoice.

We stub the OpenAI client with a model that *never* stops calling the tool.


In [ ]:
# Stub a model that never stops asking for the tool
from unittest.mock import patch
from fastapi.testclient import TestClient
from app.main import app

class FakeFunction:
    def __init__(self, name, arguments): self.name, self.arguments = name, arguments
class FakeToolCall:
    def __init__(self, id, function): self.id, self.function = id, function
class FakeMessage:
    def __init__(self, content=None, tool_calls=None):
        self.content, self.tool_calls = content, tool_calls
class FakeChoice:
    def __init__(self, finish_reason, message):
        self.finish_reason, self.message = finish_reason, message
class FakeResponse:
    def __init__(self, choices): self.choices = choices

def never_stops(*args, **kwargs):
    return FakeResponse([FakeChoice(
        finish_reason='tool_calls',
        message=FakeMessage(tool_calls=[FakeToolCall(
            id='t1',
            function=FakeFunction('lookup_order_status',
                                  json.dumps({'order_id': 'ORD-1042'})),
        )]),
    )])

with patch('app.llm._client') as mock_client:
    mock_client.return_value.chat.completions.create = never_stops
    r = TestClient(app).post('/lookup', json={'message': 'Where is ORD-1042?'})

print()
print('Status:', r.status_code, ' (expected 429 -- LoopExceededError)')
print('Detail:', r.json()['detail'])


---
## 10 · Failure Mode - Degenerate Model Response (`unexpected_text_response`)

The last one is the quiet one. If the model answers in prose instead of calling
the tool, `finish_reason` comes back as `stop`, the loop exits, and the service
returns **HTTP 200 with a useless answer** - `tool_call_count: 0`. Nothing
crashed. Nothing retried. It is invisible unless you count it.

`call_with_tools` also logs `event: unexpected_text_response` for the degenerate
case where the model signals `finish_reason="tool_calls"` but ships an empty
tool-call list. Both shapes are below.


In [ ]:
# Shape 1: the model just talks (finish_reason=stop, no tool call)
def just_talks(*args, **kwargs):
    return FakeResponse([FakeChoice(
        finish_reason='stop',
        message=FakeMessage(content="I don't have access to order tracking systems."),
    )])

with patch('app.llm._client') as mock_client:
    mock_client.return_value.chat.completions.create = just_talks
    r = TestClient(app).post('/lookup', json={'message': 'Where is ORD-1042?'})

d = r.json()
print()
print('Status          :', r.status_code, ' (200 -- nothing crashed)')
print('Answer          :', d['answer'])
print('tool_call_count :', d['tool_call_count'], ' <- the tool was never called')
print()

# Shape 2: the degenerate case that fires the tag -- tool_calls signalled, none sent
def signals_but_sends_nothing(*args, **kwargs):
    return FakeResponse([FakeChoice(
        finish_reason='tool_calls',
        message=FakeMessage(content='', tool_calls=[]),
    )])

with patch('app.llm._client') as mock_client:
    mock_client.return_value.chat.completions.create = signals_but_sends_nothing
    r = TestClient(app).post('/lookup', json={'message': 'Where is ORD-1042?'})

print('Status          :', r.status_code, '-- the loop breaks out and the request ends as a 429,')
print('                   but the tag that tells you WHY is unexpected_text_response.')
print()
print('Fix: the Format directive in SYSTEM_PROMPT (llm.py) is what keeps the model')
print('reaching for the tool. Delete it and this failure rate climbs. Alert on the tag --')
print('a spike is how you find out your provider shipped a model update.')


---
## 11 · OpenAPI / Swagger Docs
FastAPI auto-generates interactive docs -- try endpoints live in the browser:

> This section is identical across all weeks. Do not modify it.

In [11]:
from IPython.display import display, HTML
display(HTML('<a href="http://localhost:8000/docs" target="_blank" style="font-size:15px">'
             'Open Swagger UI: http://localhost:8000/docs</a>'))